<a href="https://colab.research.google.com/github/asegura4488/MecanicaClasica/blob/main/Noether_friccion_animacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simetría de Noether para un sistema con fricción

El sistema está descrito por la ecuación diferencial

$$\ddot{x}+\gamma\dot{x}=0.$$

La simetría $t\mapsto t+\alpha$, $x\mapsto e^{-\gamma\alpha/2}x$ deja invariante el lagrangiano de Caldirola–Kanai

$$L=\frac{m}{2}e^{\gamma t}\dot{x}^2.$$

Mostraremos en **tres paneles animados** la evolución de $x(t)$, la disminución de $E(t)=m\dot{x}^2/2$ y la conservación de

$$Q(t)=-\frac{m}{2}e^{\gamma t}\dot{x}\big(\dot{x}+\gamma x\big).$$

Puedes modificar `gamma`, `m`, `T` y `condiciones` en la celda siguiente y volver a ejecutarla. Requiere `numpy`, `matplotlib` y `pillow`. En Google Colab funcionan sin instalación adicional.

In [1]:
"""Animación de Noether para una partícula con fricción lineal.

Ecuación: x'' + gamma*x' = 0.
Lagrangiano: L = (m/2)*exp(gamma*t)*x'^2.
Simetría: t -> t + alpha; x -> exp(-gamma*alpha/2)*x.
Carga conservada: Q = -(m/2)*exp(gamma*t)*x'*(x' + gamma*x).

Requisitos: numpy, matplotlib, pillow.
Ejecución: python animacion_noether_friccion.py
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# ---------------------------------------------------------
# 1. Parámetros y condiciones iniciales (puedes cambiarlos)
# ---------------------------------------------------------
gamma = 0.6         # Amortiguamiento [1/s]
m = 1.0             # Masa [kg]
T = 12.0            # Duración [s]
n_frames = 70      # Fotogramas
fps = 14            # Fotogramas por segundo

# (x(0) [m], velocidad inicial [m/s])
condiciones = [
    (-2.5, 0.7),
    (0.5, 1.6),
    (2.0, -1.4),
]

# ---------------------------------------------------------
# 2. Solución analítica de la ecuación diferencial
# ---------------------------------------------------------
t = np.linspace(0, T, n_frames)


def solucion(t, x0, v0):
    """Posición y velocidad para x'' + gamma*x' = 0."""
    v = v0 * np.exp(-gamma * t)
    if gamma == 0:
        x = x0 + v0 * t
    else:
        x = x0 + (v0 / gamma) * (-np.expm1(-gamma * t))
    return x, v


trayectorias = []
for x0, v0 in condiciones:
    x, v = solucion(t, x0, v0)
    E = 0.5 * m * v**2  # Energía mecánica (solo energía cinética)
    Q = -0.5 * m * np.exp(gamma * t) * v * (v + gamma * x)
    trayectorias.append((x, E, Q))

# ---------------------------------------------------------
# 3. Crear la figura con tres gráficos
# ---------------------------------------------------------
fig, axs = plt.subplots(1, 3, figsize=(14, 4.8), constrained_layout=False)
fig.subplots_adjust(left=0.055, right=0.98, top=0.79, bottom=0.22, wspace=0.28)

etiquetas = [rf"$x_0={x0:g},\ v_0={v0:g}$" for x0, v0 in condiciones]
titulos = ["Posición", "Energía cinética (disipación)", "Carga de Noether (conservación)"]
ylabels = [r"$x(t)$ [m]", r"$E(t)=\frac{1}{2}m\dot{x}^2$ [J]", r"$Q(t)$ [J]"]

for ax, titulo, ylabel in zip(axs, titulos, ylabels):
    ax.set_title(titulo, fontsize=11, pad=10)
    ax.set_xlabel(r"Tiempo $t$ [s]")
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, T)
    ax.grid(alpha=0.28)
    ax.axhline(0, linewidth=0.8, alpha=0.3)

# Rango de cada eje: fijo durante la animación
for j in range(3):
    values = np.concatenate([data[j] for data in trayectorias])
    lo = min(float(np.min(values)), 0)
    hi = max(float(np.max(values)), 0)
    delta = hi - lo or 1.0
    axs[j].set_ylim(lo - 0.12 * delta, hi + 0.12 * delta)

# Las curvas suaves de fondo indican la solución completa.
# Las curvas gruesas van apareciendo a medida que transcurre el tiempo.
lineas = []
puntos = []
for k, (x, E, Q) in enumerate(trayectorias):
    tripleta_lineas = []
    tripleta_puntos = []
    for j, arr in enumerate((x, E, Q)):
        (linea,) = axs[j].plot([], [], linewidth=2.3, label=etiquetas[k])
        color = linea.get_color()
        axs[j].plot(t, arr, linewidth=1, alpha=0.18, color=color)
        (punto,) = axs[j].plot([], [], marker="o", markersize=6, color=color)
        tripleta_lineas.append(linea)
        tripleta_puntos.append(punto)
    lineas.append(tripleta_lineas)
    puntos.append(tripleta_puntos)

cursor = [ax.axvline(0, linestyle="--", linewidth=1, alpha=0.45) for ax in axs]

fig.suptitle("Movimiento con fricción lineal  $\\ddot{x}+\\gamma\\dot{x}=0$",
             y=0.965, fontsize=15)
indicador_tiempo = fig.text(0.5, 0.86, "", ha="center", fontsize=10)
fig.legend(handles=[lineas[k][0] for k in range(len(condiciones))],
           loc="lower center", bbox_to_anchor=(0.5, 0.035),
           ncol=len(condiciones), frameon=False, fontsize=10)


def actualizar(frame):
    """Dibuja el estado de las tres gráficas hasta el instante actual."""
    i = frame + 1
    for k, (x, E, Q) in enumerate(trayectorias):
        for j, arr in enumerate((x, E, Q)):
            lineas[k][j].set_data(t[:i], arr[:i])
            puntos[k][j].set_data([t[i - 1]], [arr[i - 1]])

    for vertical in cursor:
        vertical.set_xdata([t[i - 1], t[i - 1]])
    indicador_tiempo.set_text(rf"$t={t[i-1]:.2f}\ \mathrm{{s}}$"
                              + rf"    ·    $\gamma={gamma:g}\ \mathrm{{s^{{-1}}}}$"
                              + rf"    ·    $m={m:g}\ \mathrm{{kg}}$")
    return ([obj for grupo in lineas for obj in grupo]
            + [obj for grupo in puntos for obj in grupo]
            + cursor + [indicador_tiempo])


animacion = FuncAnimation(fig, actualizar, frames=n_frames,
                         interval=1000 / fps, blit=False, repeat=True)


# Evita mostrar un gráfico estático adicional en Jupyter
plt.close(fig)


## Reproducir la animación en el notebook

In [2]:
from IPython.display import HTML, display
display(HTML(animacion.to_jshtml()))

## Guardar como GIF (opcional)

Descomenta el código siguiente si deseas producir el archivo animado.

In [ ]:
# animacion.save("animacion_noether_friccion.gif", writer=PillowWriter(fps=fps), dpi=75)
# print("GIF guardado")

## Comprobación analítica

De la solución $\dot{x}(t)=v_0e^{-\gamma t}$ y
$x(t)=x_0+(v_0/\gamma)(1-e^{-\gamma t})$ deducimos

$$E(t)=\frac{m v_0^2}{2}e^{-2\gamma t}, \qquad
Q(t)=-\frac{m}{2}v_0(v_0+\gamma x_0).$$

La energía cinética decae exponencialmente y la carga de Noether es constante para cada condición inicial.